# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJztfQtX48iV8F+pob8J9owx0I9JhizZpcHdTYZHL6Z7NgdYHdmWQdOy5OgB47D89+8+qqSqUkm26clmd0+Uk2ks1fPWrfuqe289boyTeBre9ueLjT2xEc7mSZqLJLuO5Z9pUP5ZFOGk/JGHs+pLdlfkYVR9u0sDfxLGt9fxNE1mIl/M4YeQX8/neZjEftQTJ2GW98RROIb/HsQLWXru53dROFLFP8LP6/g6fiGOgjwY52IQ34dpEs+CON8T75PkNgrEYRL5I3GfiZNk7MNAjofe4fnJwVuxD3PpY4v94FfoLetcb2zDjHOofL3RFX48EXGS1wsd7l3js9HFrifBFOCQJdF94E2ShzhK/EmQepMw7XTF1p9Elqd717GA53qD/kd/Hy1ifxbCcKKFiGBYeZCp4R6l4X0gqqYENAVTS9JFT2TFHOeNAMvu4I9xkWc9brGvXmzlfnob5NnWaLEVTnhsBUK1RzPK7vw0mIhpEkHbmXi4C6HP28JPfZh4gC1P4U3GjUJRcTb4PLgQD2mYA2DkYGDUeWIOOE2SvF+faDglGCqYS1DgM/KzAMGkLYM/yvDfjvoNXwFMQcfzcEye1+1W1dMgL9K4rPpLEsYd1WQPhqCtRZb7t4hwvGBY+YXY7YuPaTjz04UY3wXjL3visEhTWHnxkKRfEAwl2EUHJjFLCoDPRBQxLok+czmm8cPEPRNYC/jWUWOHtuBnH8aU5tlDCGUqrNueYIPbCvmwSQReGIsru2G7EtTp1XqvtXy6OFq98JAwhf7OoMaNtnZ5utB+4QPV0hCQeF9cBf0oeQgQ/adJKgIcPvQVwebBTQGT6t6YdQEkfrzofMGSqh2sSi+uYCn93L/ewEVN/Qf5RzAN42ByvXHTFVASqwf9IJ4oiPbD+SIeISDLMciWu9bANVyCodUGZm1+A92gPGFaObxxVGR5kPKPmY9/93/JkhjGsWq3wa/jYA50jP6BTWtVm/tZViHxy744RORt3PwdY4uqQkIWEknMxFGipnzt4VaGhawhREMv2la3oKU32G1FH1ylHAZs4kpT/bIdD3vTth0ti16rh6126zUlWVL1woy6o+ZcHeGDsAjjInAMozadcjVX3RWy6xt3M0vwkCt/FSqqR6Ikt9g4mH/sbl1jrFkxqmFHBaySQxBnWA3w0CITZxN18PXyoUKpepklO149dfRbi1S8UqRi7MdJjJIH/jUJYbWCktWFiu+Xnwh1q3abGAq8RRlrYogusPar1tyid1vPqOnRO2+9mvrKr1debImhJILtVXXm+Qzo1KuvBSKjuj5briPJDG62MW62ark1HKrj/thGcTfhA/k+0nYd/iQBY+ymw1Rc31GyN3zv2mr0vmGTaGNWYg03s6oYtZaEZAlFdTjQpscBWLLS8h3Pm7dhagyH2mpofY6rjf+6L4aBn47vhHumhAVAkaVGQHTGD2OSgIGSb0tGgnR75ufjO/yAQjm3P1swVjukhRKSTdKBrNsuGFQcVOOWqiZM7y2MLhPvPg0HmsA+9kGmrwsY2NZCYy4O+E2LKPLmNs+Q/fW4hVXFCW7rtxQnlsgB3OHqcgCu6brt/bUIiqA/Ga0mR3ALz+Z59V2AD2KfhxSGVkOJVe5FoaKmsmWwfQQBvCMKe72BuCFbR7pTldU/NU27edkYIG6iKJGkueJvTCG1Zv/nUMoKSA0UUz0r4g0+btzRerFxcy1x6k1JVZnVsoYlO8w07uuijKZi3UQe9VaeQSON6r8hLdTb/b9GEP9J0NaYMigzdezQIN6uXTk1K0AOt27VjDWVZvXsvXwUTP0iysXUj6KRb6hIlWYpTbyTFkmnQcw37KA1Cs6t6rNbpae6SK/VqtN12Uk5krLw34PQOwtZDGD9OapZSgbB1fH10fnPZyfnB0eDC+/o+AIabDLGX8cwc8sOfXJ+eHDi/Xx+8dPw48HhwBxOHszm2HUQZYEs//ZgOJDdrGuzrndlbBzVMpEodQaBP3gUXT7lGOZpMQYQAOQMy96RrsBfHPxsjZF6MAFV2meg5YvBu+OzwdGKlZQtR6t4ev75eDB01deaJnqb3IdkTK7qXn5eoV5+72VBalc9v/xQrnpb7SS/k1To6ODyYLVpMsOAKqeDywOq9ufh+VmtnmyP5hbkPtZS3IRXbDC/C2ZBCrSETp/E51NzsRgtGpbMwhm5Zl7IzcuqNNdVa+PiqaOQy4uD4QdveHnw/vjs/YoN5Kmf3RkHKlyoCbCOJkrI8rdW+JqNu6F8OTj9KLu2eoP1vjj+rGBjrrE8NQQqnIajAk923n/8JA6lsvs7cXhXxF8y2HA0U3PNWkBWRyQdVocfPp39NGwDufaJKAGNQoOWbGE1SGu1D08+DS/dm0WHbikcaVWwycGFs0+tVayNJ2dBanZ4ekD/vDs+qZE8q7Ypi5UTbh54HTn04ZvVWyZR68eeilmgZULOluxp2WinUO6QjtqL1Ed5RaKKd/Tp4uDy+PzMGw4OobeXr3f6O6UU80K8FjOQmNFKO4cWvgSLaQqsR2TBLR6Ai04Cws8MiM7Ij/x4HIhkKljUyZMIaBK++p0AcVNyEbKiHG+fK0z1TgbIlS6PTwfnny7lGF7tyDGAElR2HwV+BjKPHL3ozNPgHkaQoUwFH/wRSHV8Kp9hKdxRr7dmaHlEFAUB34+zcTIhTYWXyvswOLi4fDs4uLT6330D3b8Qp376RfAqwbSmEfAkEh0nQRTkAYtb4g5UtHwU+DmpPLtvYO+eHvyHd3B4iWRBLdSfz98OoeHXFVzl6bxSSZICABzOA+xkT8DctkbFdIqn58Ucz8Ffi1+SER6lBzAhuZxZlOBcgxkOCscJZOPj4PDTyQH3TfCF3X5yIuf1muc1DKDTSYZLtfMt9JXcgjyTCaAiEaxjAPOAeUqwhZmAMd2Go4iteNk8GBcRLAGs5dwHnhMFEYjC8JJxClHvc5COEliSk+SWCNvvQPwdFbfiLMnDKci8WDITl/A1Ak3saPD203vv5Pz9cE+MkgR1FBR0AvaywKNtEO4GZwdvT4DKlmVZcNlFbeQblKh2CPGHfwFon2IB2jx76BnRRouyBQByBtrQbbVvgPDjGT/5fWR7pgykHDzKA/xxMl8oFvDvnwafYIhv2zrUTFtq0zdVq5MeozKO9VierfaEHz34C8AVGHdEAxwOj8Tw309CwFOUERGFYB1hO+BZjDEn2pFYZgtq0qSmfhiB/Jf9UZBU8xBCq9iyOU2XrKvNxZycIeFqpewm2delmONBgS1dx8ED/kuLSm4vZ0msmmR3EN3jZQ4whN2BKC1PpoG1+rcB6VsZwsMX0CKUnIRAF+4DFHhdHjF93dXkNkqA0AmbBUvZCv7QpcK66Fq9Y5G0+l3KmWY/5eobIkxPlEJHTxjsvM794V3FLIzGHXyrKqyxIOjC3FY9a9l6Ql9ATROVK4b2grqlRn401VJUvVQjNa3LVodUC1x8bZWEKj1XLdErP0c10es/Qz3Rq6+vohB011dTsNpXqCrUa4uwjN+fLfZi5a8Rfan++gKsXu15QqzewvMFWYKduUvX4HvVyqzNw7Td7vC/a6X0WABFCjYRVZWeTVyrJpZS2Z5GWtchoD3XassD71b7OUBz5n8JgFxlnUlPkG3OS77sX6aF8u7DZ0WjIuj8cQYsGijsGEQ39DDU/Cu49evYBu5XAva/F6g9p0mrJwxjRvmzNFD0RM3koApVg7P13PKNPqZG5e46VkttLvMKS9yyvNXSgpY2Jj/R8QJE9Zk/vwOBPAMp+mdWRU5AjMulmqGWVI0PlSf+cnL+swfaAK5qvcwOaiA7gBqTkP2CP52dHJ8eXw6OUB8ZSy9V0Nq2QN7X5CLZqwR31V6DxkPynKwK3EtOBcqXDtr9coId13QAbDAIjwbhOVtCGRAnQ7KgKGKUcVHlHWtAJL0FJwMaKkjCaTj+EgX6nKAwyH8ZLMhKozSnL2Xxy3AWJEVOCuKQgPYzujJMkltxUZCzczk5wM0LW+X8ETRerUi1epcfLgbDD+cnR97bjwjq3Z0d8R389+XrPq0i/v7p7XbmrmyqtS+pEwPQPJaLweUFbHZU/7HN4zjMQ5AQv4efIJPJg79XIDTneEqco704z5QOsjULZugRcelnX8Rl6kslAoCOh6xb6KgvIpTAZwm0m6Tkl59DYXL8upF/e6h9GEAHTesLmtTxszcm32hU5na43/csxEr1HLXoNIkyDd/4i+ePc3YewW14HbMMt9LHxgGxkpL5CKW/BWSDJ3M8/qfSTmyn/KEsnwlVgZ2j0iTLtuagUsOvGUxkBlw3HIVRmC8ImUz3Wn8a5KZKomRt7NwWpaHr+3ASJKos9YrHF/2sGHXSzSuMMdj+7l/3rjf+5U//dbPZo7H2qFzXXeX6OvueyomyYB8tTXN1mih7xi9Xe4ByNzhCmDCBpyv+hGgoUBuULj4MzdECCaDHtl82sHTocANlEA82Jh3cVNBFS4Ej5uEtNWMHOlxiq+JtqOjtaYKn5iAOs25MUR9GBVAdE/oeKYN6UNrYETtA/xhj5TTLJZ/K0TAzD9IZkJgYYxdS2BTYS5hzI9GiLy7vwkyUdit9bH8tYGshFmfFjDhDc5hDHSq2fiUP/uoF67qWeIfGs2XczMFSmwSYIg5BPvQk1kyVNf8RY3f6+J/XnW7/Lvj1au8PN0/eoxoyWlxoDzkG/aQAgA/BEhAky20R1TFIbTDaEF/Q+qNFz1z0BBedF5zCU0hGrqpxjFEfV9Uxyp42MqOvT8CS4i+MY9z4JMy+VCXqh9RaQ3UhktGq4+xMLinTMpe4IXyQDXWZA2hx3oFFuiIs5M1zA4wrxQOcPZBEiojDQ3h7ClpMIluPdRA87YnH4Mk4ia/LwfXptqNpfeqO8vXiNVC4wOEUrq0W5P5gIjWOAj8u5tIfGH3/PaBH/oJYAAh95eu9MsjsCgrcSDml6yBYw7EfZ/UoJ9Fxn1iz9zp1KkAFw51PSqARUDUeA82PczKDqZgq1P0DWMYqQKMK2hJhlhVAqqAxMgbe3iWwvcbJHPi+JG8U1QW8RyNs0OJMFBnyegfxdtOvCkJoWCbZTUXAObRHZQ7CvysnUK0NGPAKLqEO0uhwCzVtTybitjqILpXszZZfoGEZJJk405ZyoozOCA1lOdHXlNFsrmoaDvPAmlFqgMKd9HrjP1k9/+76GvCj/93/Qy4NJY7fn51fDA4PhoNur7FqabPB2ohWbfVvNO8WHC3wtSxI7/l0Rp4iyA2zpwoeTyu70yQBiOLa3PmwrsqoQD7AuukIe8/GRUDseIR0v+LZiASu9UK66E1qYRmtRoxmZJGt2VSJtn+9k9KPt6EP2Y/tpMONma5JGGEEAmP18U9ix00aS05dTbGJQatHsjM8wJjS+nMnPQU8R5WKXRzyut6IC1qbibj3o3BSLSKtjkbNgLE+ynaftBDJcq1wxdc3Ji5ZMqzkXDRXb9qyNXfWsnbUWfPq8ee/+/phN70KpM9ZQ2P2zQuJxSpev7qgUfYqhQyOjK0URtrqRPgqaYLbIVpC7lw7le2wxcWU3QT9euxeo6d9zaGUAFu2Y+NS3ZkURwQlcTwmwa4LQfClT7EOHR6FKz7DtVR4hICR2cQ2aEn8NA+noMgivLCppxrK4uPUr8qZ1csraH+/L3YdrYFK/KVksKqsid6O4R/KgsVcPMpKT4oDjrYrOcahkXHotzr4vChinrxkMBhlOksmBRSXUfvogVjEhGGhPCV1HVgukeigO2B/G1FyexukjSkKQFNwpyXAU0zjRXtiAisdQa8UIqWFiSpvoRgGmsRteaD5O2X9uI49eF+3W1wowwUs/Ki49UDTGkXBpDxuRwqD3HnA71FKnCh/VdxjfoYvoCaetvTQdRUYMZACPL+foB4FLafBGLAP+4eNixO5wnmg9EtzubmRth6PbZPvvQ/Hw8vzi7+wt4cceSnau+Xn0vIS5DyTjjETU7zmyaA4AfoW/Xkv3RKoKkKQnBN8MsDQOTFjUOY68DUhx5/QNVqUENcw3ywMg8fRqbFqe63ycrhyejjckK/oLvpRT+1GyYBCbg+t2hYjFqfYqFEvTLtR9QeduWaJz4qnErg2YSaXxm2luSBZOGOUg1m51wSdTngMxlpIlcweZYUUCoc6DhwysUN9An0mwyh1H203GaAt2r1JwCTOgeiPKC06Qf+2j1TGyMrgxBQDkZdhilEYVoDr8Iw8NAj6uYcUJMv92dxO9iHhoahMv/wjTh46ZJWb4s/O9ca3H/a+Pd37dqjlE/GQ9KUempwS6B/9jjomjH4O50CKMWxvHsRISMT1NUbQj9HgyuQLRHLgb+hVEglyjyJ6CkCVu+p8KJRl09xWJqp6SNM8VFVxsNALoPz3bGIU34kf39APer36OQrNUWs3D37N2XxoWWVJqaUy5BUCyl1SAAH2izwB6EtvkjSAeYwRBkUcxONkQhTlUxyiRxnIMH5KmQ6i4NYfL4QEatuUgWf0uat+Nb6u8/s0KrK7jjnzjux6ENN/0zQBOns+pD+WBfKMAc9AOAXRJO1U3aDkSzOjE/UeGyxYwy7y6dYfdEJDA0TQ4qCR1cA/faocAJkbg2SF48j20WUC4YYu+/1JoL5b4kYNFGXTzQUNmKxJoqLgPoi8bDEbAVWmHy6jfXSPrJE+94v5vIqWQWs2fkPafHz27vx6w2V2vy5e7r768bqYBjtTBbog0usS5W2qPPnDqwn8d7IzqVUGcQY2yc8HF2fsA4d/HZ9BSw7rLo3jB3+nbRyDi4vzi8ZJ/P712F1t+OnwcDAcNlfceWPRbXq98/Ilh1rQ/gxmIVFtbRl6Ivdv5V8zPNu+DeQvoCJjIJQoVpaiC5nnXHa1QzQ/KH6C3aC7BH86mCM9y6TuAFs+LI+w2Oeyp5EDFLSIH4TEYji/EY2BPsud7rB2NdJ8ZcavZkNtEgQMzChNY5bYZmm0hnUJnxztQw7mYdqzFlDo0WzqeqMsDcsqyrRP1XcaJH6jP2qfYeWoon9b+ySXEj/LP+u1UVLG6sjCiHFpQ37SuIUma/Z9WsyOQ5XDIye9KKrftvRpwdJoeZ7MOzuGqQLm5bFDKWo1j/Dz6YaFM/hTSWQ6neQ1yEkG5CrZ0414rBOg7pN4lK3DXxJAT1VMEj4vSH1Kgc3cJooZE8+VtjZCJYqcDx4IbemYSQObi+G7eXA1bma8sWWzQXHUlNdNOLqDKXU7Ry2SyWiuu9z8QdsrmeMaG1WRIlIUpuJl+yX/6qJVYtoQ4zftt01dL7lyZgWmcTE6PS86bqJmUq4P4e3dFixjkuLZb6x5S4sZqqM52ss70tWXljvA3Gu3fjqJ0IE7mUo5mlUzUzytaK1iW0Rpy/EY9FVBXYrWxRi9jFaaw5DLrjj65iGW7GXlUbLascoYj0D4CVGJmoT+bZxkIOTRhukomCYowoYKmpU20jxcSbLbBsusSo32wU/jlQar7GS0m02wrgxMJSysCEmS3lYaHMmbXzU0KX2sPDYQXHUq3SF/nz08sGffGJLiXPYHpxrKLTG1lGc8PgZClBGQE2jEFOMb+bqUcvB4yOA7V1s8xpumOZCo656I7UOixl1RqVkR5SGxAXTCADxJpo5ZrTaD6pjLOf6aCOMO6y+lvTOWnmIj6gMaTtIJDHwRoD2vqopzUA5BJVG2MnXZbMbkx8HVJrzZvFFMWf12ceayRyVAlC0oCQjbsVg1fKbfmzcmv4b3EnGhksEvKskXGQmbn6nbrn6knDIqm1uLmH2myaYIy7swq4UkNBsXdGmG+lH2zLsgAoVG+jmtb9akcxFnUlYrD2uRRlE46qfBX4sAHVLN10Rm7JdzmLSW17UYzdMEGcqKJlSH8ZTTvF5XI1bWMPmTjbv63rydF14YTxO2t5h0xFij4xjDoUDeZl8RcQec+AHzmSLinn0+Pjo+oJBPLGXuQWzeFMCvN7BbcomC5YD1h3rs8Gml4bpP/Zk3G2GpHeMD2bBTrq36eXLZHtBNcV+DbZ/OYDx4PS/yjrlJrq434vsQGOVWNgtZ4dzaguVMF1sw3n0cb48RtE/egT0ehSf9KVUNJlj74+y+Fyd3AUbYwB9FjI6sGzeWKoD0cN903bB8zPCZY3oKJBhz9Y3PXyjrYZH3s3kUYuhYD/MC1DQDqg0qwb54ZdtMYWmu9NW4IbscFL/asZOKctFyRbAkHnpw6d0b++iQi6t1qtp9ebOiWUujKdiYZnSNgQePPf8eBBsUVRrMr5SeL0MYnH0enB2Ku+B+vH338ofXLC5juCGir0wBTJIQbDgQgCSpaLZnteMUzHo6nc2DW4kPqjdc+2q5e2h8A6qwrzVzNPh89unkxEVTcfA8b840gkMgexVOyPqwFL62u884Qn+nD0QoeRs6jAwHBQi1yQxjN3GbX5D7aKr8tGM0DVaHUFk4wbBKzhTNFBh5YwCL9EXaD45kJFomTn0Y933oq1TKR3xiCaMKZyok9KXuq01unhyzqdR2BnJGzZSUidadrRipFCbQAwTPAycqqDczMjDrI3DYOVCjPjw/O/x0cTE4u5RO9ehIrBRGMvQAPQWRz+tkQTTtSWzywon7nAfmmYCQgLGKSeoBY2h3p8IH2+0rus32TY2Ma+XQ+wm/ILLqdcwd35d2SyAfTMYZb9GMWX25DLLIrz4oIkQoeAaQ1+UN6qucNnRe/Q3lpyokyHssB6hS9Dw1+E7WWkd00/wuGXVF59Gc5qaaJYgq3VobHFbkBrm1Jmq+/ZT/4AQsdBxsFuQ00+7KLJrx2Z4xELltjCNKqwj7bnsUBL3HXBpob8X3bQUA5/C4sycxbJf/eGpptNkB3KgyJV3S463nwUQjdEGXIyJJZBolfk7da929EKTuQ+NJcXsHRFJcHJxuoW+okHFtexR0uz0J7rezuxnlxFXknWB6B6oK8K83Ozvi9K2YpgEQbvRL1rF9pg6WYHlkQ7VFJx8+D3kY9b6v+wQT23KkDeN2nZmU1LeVbEIwVeIb7MyC/XsFCtJVM/U6aImFan2asQQAR0TIfxpMPDRZdpzFAAYrL7xMPDYrrUiakNyYu9cNPxNZ1aMFQD9jZOrAhoVXHJ+diIQIkZVYqLtsIisYs/4xg3NA6x8zEN1q6RhB3VJZcT0VWO5VZJWaINnMIK/aTHXeis/g1zu/yJAq0Vk15gOUybBJRtAI62YmtLBQMQr8MXpi0Ums1sFuX539u8JOmVjT58acJFVbmBy++UqD/rae3XVbGxwTDhB+KYVIv6Vg1RdmlybZyQyIKUbqmokGl+nt71ZpHVPYqtsuai01JKVfrWVM48gilMzWaLduJFJeqcl3aB3L9PwNFdTR/sQCJ/nzsTkOxDgZdsJ5ayfbszDL8IoOcrdSOfr7zWhYpm8m+qh8fKRocHVjHlTs9mWqGZnun3MumHYd3Kcym7sLxTaqzev8bKf6nGuH2mr34w95kG0pYlMxd1A6a47KNDTvmrMDpOckIGlQJkIxptfQTlM2N6Jcq6aP7H5tN/0+t7tyd/rUYQ/+FIN0Y+V55utlyL4XxsaVOIYUUafOyhsP+zaOctVT+eVZTu8Vnjb5//dWKGxmSV2lxpL7KdRjGQooEpvDHmJ9Tg4kbHBobrqeghDhq26N0EIauBcz47Ujz7U+qxVyvKrHmcxzwjdoNOV41aCyRsLOEjDtO4RFPmxo9Q3hkOvWScuqb6ajYGtSzCOKwFbRE2TBTzEfpSajBzFpq7muesjQNvMSBct0Pp4bifcl5TaHF2PQaT21ythOF01oCUVl6mYclFNWDOK+P5l0sKwDVrVRqyWR5c3xS+YB3dXq1dx7UJKwJEJ1tszNLHXjr9pwXgPTuJO0/aM1wWHUkgUbqZYaNtSzs/s2XYmz3iyq83SuTflunnGOjg8FL+3TMUEfQ7E704adjdaBfSou03xVgnLbdSnKqADSRwf+7pqpiu/yfG46Q9kPOSl63HnZhmHGaK77QpyXSbfC2xgdL3Zf/r6/A//bFdsc4Unhc35GkXKw9jLbC25E3D0pyOV3utyu4oNbqd8qvJOAAjMoB8Tmz2q6ZJoqh2h9XUpMG7lMBRt5+o+BBUZeLV5XqZJ8I4aLeHyXJnH4t6D0Q3YkA3I9HKbk3OfLfstt4aLhGqDLDlzJ8OW3ZaBqT9OvARRn7k52VnbV3s4ayerV05y0Hh91uYfCit+IUMlmlRc/vkINGXdEEY8DL084BEMaiNG+U2Qqfd/1RjiJAnNPS+cI9tMbJxGQYBjLFtmEZE4IqQCT957DoK2rkX1b3ZEkrOYgV5pt8ZjNNOrW/NlKg2xZtnxTK6uf/bXYpWv1tMNAu1p1KlWrxeClSvRX3c8P9qtXZra0nPJqpdPgFj3nAaherXBVVrN9qrvhahJ9ncq58u9s9EwZUpEQV44uM92WPfYme9EavbZkiLX6rrq25DSUHZAUS7Cs7UyXO+1Qdo0HcpKz5W+uqx9AGEj95A6CpPZKiYFbptk+Q2IgMWFSzOYd3HQwlh7mJQI5av9lA/Gb1t2/lwNMA9w0A/7TgVYwD0vSaWIIX3URyBpegvg3uYLQiSmwHS9Fb7k08BR5TNvJonXUi49Nzz4HaTgNKWrfz/Uj3c2sJMImxWSWh6YhJ70sWz6eYoqTBz+TaXEnMlZmtDBkHGIobJeSSXdiysVrBKNvjUE4yjlpSjibAXbDr2ghTyu3FJ3JGPcw+kTrosV49U96U5lhMg9Nf7gStYOe/720yG004V4a73HTIeE+ssFHC3VVz5p7Ww6w6s8qL6NLpaPw9caHwcnHAa3/1AgFeS0O7J2K205NAn7y3hQXAe0jfLnZBsrNfr9fAycfg9blMiI9+/xPXZe2T49dpgE81pU72MMNLI9FbDLuTN1iu2fgu3pGFoYKn7HUiOFFmbdEJ346yatHHRP1K3KUzWsi4j+JylcTjr8DGViLBFRJhWS5r+bhHlofPOnwKLl2EE8oGbX0pkYpZ6/RS0H5thhHhG5X5gq1hxTP5Mfiw+XlR/HxfHiJqv77waWQI2GGOUqThwyIB/vfcQ7s+yTE+wGSYjKN0D9ohMmeZCLwGspLMtu+2Q1VT71kYwus6aNV+elRgcfwaKnjC8zEk1qZ2TQ+akY1jY1h9AlmvXVwS/fdoHfkafK3MIr8bUyH3/k5xJyBmTi7FLs7/Z0/Cnjxw+s/il9/eN3FWLEo+DkY/RTm229e/b7/6gfR+enD5elJT0Thl0C8B5Et6YrDOyAdwTYlWMT/iaE/9dNQVmk4WTgYI17xiPw5G4JhvbcZvZ1V/mOLfcu2jo/qCqhZ4alGowl6Iefc0VKnNwC5FMszksu7Kr6zEurr1eUy4JEeH5psXS7mgQpkr8+xtsJQzvQZ7l/IrQSveevsqzH2VHf78l9rQITyVmvwkzQW+N0jLTUp8v1XO1221mWuQ0G+ygp9TJHzYYDeyx1XFhd8RslkQbkSoTz6DHXKqNc2qPHcadOUJtOsg23V41t11+k+7naOwLDSrchxIxX3w5mHCTSvN5TpviRH9ZFISYRiUyw5hCjLI6aLmgRPyBerfYuJR3C6MPZ67hEnNXBN5tPFyfPnIm+ewxWdkLtnWQLtu0FXOdOtN2dA4jgYVxnSfqNpy0yxNNtVaOey5DolhL5i+hw8Zc2/6rE28RVn3GD2c4qBpWeMU5W9qHQ/tu6VFj3a56W/LOHpwcdjwggW4t6SiFdjZjAIb+4vcLf9n7H0Oa1tq+kX42BXKCCj5lCBVVfjOzX+3bWViJTOIqmYKQ1db2z787B0bVFIwAnmyrUwg3hTykCB2n8mj4iU7bLLsdKV7dGIRMcHRLPwljNGVdXVSwnnnrVcFlJL0KkASAt2xmlHNZAG0MGu3XxU3T9tfrPM3a2JMF1v8EVhjOrcrbwTSebp+KNAAxdm9IINYXpLsSlnpHaEceTarvmV1qba7mZNrdrdGYyiMh5LIXiGaaTH2WqCrzPkpqIFLO5SSuny8iJyiC2vYgLJFuFf2/PVfpfjQTlZ98qVBa6M/X+j8Nkhaq2K7Zoxvad66dq9U1Jb+dFC955wLAB122o2VFq7XalFwTc3oHs0cvNV+ULcuOvkKhXF+fGV+KDAsse34YkXj3qPm+RVDQD/dbP7BMTH+DYf55s9AdJ2d6+/O336tiv+TZglpvMMS0BV+MuFumlgWmBLEcPhp7lMCfuYRJEM3EB9OMbUKJXDNt+mxdsfYyxqqFmKMqQmGbhTDetfSwzct7Xi35U0bP9Rj6XrY2ppPlfTyNyTyQkbcFiNaRlNvvMzKZjZIsYvHO1QFf0lGdGewmUxi45xmY2y5eJj8a1dq/wLQdFMlH6hMunI4FcgyOyXz9+CX30gmNSe1al01/eKOA8jBQW3Sz+NqoMz6vFgu4x9NUlMO/4S/2J10WwZo/vtGPn0PlxKmkPS0nbdUkb/BhR+n2RFtecUzLX9hgskvhe7T9vVV4rwk2DBDZlMxZ+TkV6fl5c266ZWL8yjAFrchNLwbbPbJD2mwfIzYxWuVO3TnuBuKXq6J8o5yN8gEPOYZWo5TiaDCSMwPycITnscoYGZJfs7zSLosBjhxRtyK8thINtFgI+DFZjOVwmZaufsydnWCmjbZa8CQq2YCQ4sa76pVdBBhcWN3y6Bc1WeaKykzhdrsusSRhK8lncLAkrX5K5kKlfsxWMJlCd9+TqMwwzVp64tzRISUga/NmGUPZl9MukEE+bXfoRGgIU3QQzeuOmaKpC235cicTNSynlnxpQBE2X7eI+j3OsMBLp8HN3BZUo3adbg5Ku5FVr+j8fer8IvHcbPQa+Xu77M4AWaETVVUcxyhEQjK0oosciG8/OQSmo43LUmrVNwibrnhEdCaQ+d9mY2yHsctlJSwGaMOlL34ojUf5BYY/rQ0DlJbZ+V2je0HvgUryaVDlBG0O5YgeL5VmXrxLLRrKzc+Ta3gd+4TczLzY3KvvhoW5DJNgRy3fuPnzjMctuWxq43nlqvQVjZOPn6zRrGyW/ajJNuCBoD0vw/NXyhE6CR9OJAXtHkyWGlpE5GvyB2z5kD2z6gcjhO11Wtc3dObUexP6m00PisYjJr2/uEx4z6EpvVduPkOKCKkO2j1RRWO6xkCzpvWLlXkTTtWfpujy72VawApZJVPDwGdAcvEBMV6r2FzAivYEafDA73L+9AplyV5T2xuGAlm9QIcyZ5BCB4CHreuFKwsx5feMN3RAi88D27g276sy/3hq+d7S1SM8L5D/LslqG9TxC54jvodSlFM20hNjnr2JLNjb7Zp954eivLSkKrbqr3ZEYQWPtHfdsyLa/aVzxMa7diA2UpjZEZYVH8XgZo1+69sQBh8ChEcqTFdrCqKxoRkDeMKX75UWLRk/dYdW0iK1fm1A0rNY0QXtJ21TrKFFNKV1lOoVd1aWuKjoth1vbDVzfE/JZRGrt98S7Ix3fEDWWChD1xmS5MyxrdBSU6REZJkNomP38jDA/0xakfRUo0mGhmPKYyOuyKeFJd8OR01ZH3sCGEYV/fBp3Xy2Fqo5n7xgK7kPvegtoY3U40DgeaFxzBKOM5rQBGhhvsTeAdd+hGBqRMBTXKiNGysNmsGzfsQGHlDt6845rBN2/z/tYDQRyu2yt7ExhTQlNCFgXBvLPbN7NRwujMFVgt4N6+NsKEQq8iN0uG72Kp+LScp10NgWqIknbcyK1D7EdaazrAT7t/pG1CFwvJraLtEZfPEj5NO6YmZ1oF3XenVVDAAvYG0b6CiG7vjSVmmJevpq+a1AillVWCc18MadpVaPVSAdwFIe4nmLusi/hpWyku25qSbB/QT4Bwf1Gal0v/UL00Y5FcA2rpK5FHz2wshkmBBKWm6DNYsctFkOPNWEGWhXQvQKxcvFIGmlTxdKzj62wa8K3FUHjVkTpuJR508QxBtwt+L/gyTmezpjGw1lajSdAh20+9mX5FkQODt0VHy53hPgBrxOXf77xpwuXOI/ZNpnlx+rZLhlkAOJ9K1XlfhdTfWHIERVy/LcJoIt69wyRSmhw7TmYzYGGazArzois5+bJQKfcpzVr7yCaauIjMULNxOq1Vg3cecHGs8Ur3Sb/3cAzjWnnKxcTfuJcP/Zc/vAFAfxh8PjRvL8o8yr6Fej78yxFZqlmK1jrAGtrbqi4a3SnJFSYdcyQAq7ufYld1CaFsx7Ef7z1QtNjFcWNrvHfP09Hza1E6LwpezfnH/LV8Of4rl0cXBIBg1wqJbkiH4u4R1ORfCT7ccjrVG64N4T5IF1MfA9y0Ph39PW/yWtax/87Jv3nO5DW8nk24XSMPGya/BfYhfy0MgL0Q8pz588XBaZVAjByakEriNj77fDQ43MPM8HgHBGqfmLawLDJK8jwK4mBs3nfZBHQYI/DfHOOBEQB3D6TBypjsYuKzDbWhNCUnLMmaXhDdk3Wq8M1+ufHbB3DPkNbqNvZvNPMdr2PNxLnl1/368C36YHpxcEvJJLw8U4HnXwLvb0GauOqoc2+JduE82NutlSt1rer1jUEWJkXqZUTA6ACko+nG8hO2//I1HovUM0Gc+EUMepGkyTJhHxuS6Gy+Oo6XFz/X7K6NbOUHfxfkHrzZKtapfTJtYjaoD5kG2M7mYzkdeQCFJ0/kytScEk10TDGj5t2C0zQTHX4kqxlgQ0/eHaKnL/x4/HHQktbQSH44KqbIm/d3DVBzKN6IEvmbMoSGikWaevOxPMGy3k/nmfM93+yOe2Fr69rINF/XHOgeHEwyDIwIpyCv4XBJcFhqn/6pp+xUD+bg5AJa3DaMc785brs9qEubKGMyNy8zgGKzV7s3ajwtIbXShEPeApTzTrbrrvFVAWLUQOSEBK1MCyyM5WubaQMoeZLUgDZN/r3qGNEShGjoFc9fNXSnpXyprjm0rBIPl+gWNLEtdr2dnR38v0ugNirxHgFe1fnxx/5OT8z8Xzs7+EenbHObXu7iS0kDu12SjSURbGpfQhV60GCKv34jZNNqrLhy+Hw1iuITJw9NlEd/AE86WHTLoFmUYxcd61u6hqLamZ7lP+bw6NefpUeL9QorHZTXq9HS7pULu7Q8LexeuaxLy8tl3dMWdXmdMpJb98ZqrvXUgsGkJIQxNBmPgw6vSY8y4LOZkN8oP8oR5illj5rWZBv4rOaYEkx2RU3Lb+D31D1fVWi4pz6EMWbgLVPadvviXRoEdNINbTYp8/pD/E3eapY3Irt6Vkv1QG0++EDf1Lney5rjkP08J7UDdvMljKJlo17PNO96Vps3Pisa9puedrO+/TwDaPi058TAxxmVaD9L3AlQql0VwZWAvyceFcFhL0d0clQkZa+/MyXfRvFfYoj0QpUm4uG+CLYcqylXAuFu4f3siaDGtA8EuqY7qUfjs7vNHNnmlo6i1Z6ph8G0WqEbd4FcHr7eZJl5seoCxACkHw0Lpaiu+yj472AlXN9C6DqPxgfdexFW/J00LNCNd1pN4kgemk3i9LXZJO6EPF8y1wTeSvmTJwRr6m/iCoaPdz5MAlgha7p4U8j/pgVTV4CpEVF8MgchaBbQFg9WvNPM0XuZJFPv6jUm8ZwvtPTwbF3Xj/f1I9CqbttFX9bhfctNX/ZJVYlhPdsDwDx1lp6JDqO3Xa3Z9N0e99Fu9zaM0wQ5NIUrV9PSHL6uHUIMfTSeVz4VtpH8TZ+dIBe1kyELN0wH2ho21MDbK2Fq9fhDn2/01lxJ8EQ6Sf10wZlltWVZS+JwSxdrSBLNUsMzkjc0xLnI+3PIIcKLEthaMkuL5r3jSFnFbuqcVJ7P8QWeOsE2oqticNk7oIKKl/Q3us2l+KVb95ubRLCGIV7osa87Pz3cobeekUa/Zvde7nBHy1CdPr9yiBJVRjhr9fysPCbUQiocjvLo/eRY54aJlRNEE6M7XX+TZ5pd+Eou0o3026lXWxYNVWlbrsNBeQueBILh+IXdlUiyL/9tyq/0281Sd6P/um7QfO/HC0/m8dnHK6I7mR5ZTmidcZ5Qq+H+vR8VQea0wkm8NNtvnesKwWqN3cg1arJcaajvVNVa0vlrdXf6b2z6466n4fz3IA47R63vCyiDYpj26lsQPVyyl3ralJOa9HsM7e6Jn0H6RtGXqFQZyqRfNdOkUFuUoyKaQJEa8rMQ15JBPAE5M4bkpZjMQVfBkwYciXHnDdJLWYFxy6aQzXYHMjMC3//hD3Wuv8xM8YcdlYge+pfXrOB9L3xiQUqYGW1lO9219ALfJYM4PlLt6F7Fa7QDo9trFzHsjzKOGL+dvqVjv+5aPbKT6Qd1Unig/E+Ry4rHzb8E2aY8ia7fVkUXwWyeJaKDJ4jvpJtcd3O9OR8q5Bgv1Nyd9xM9SfQFzosXKPpxkBRZtFC8dq0+j+iS0JPkVsagyEzoOOPB2cHbk8GR6HwO0lGSBV2av2wQwMB7UU3+6Hgoi5+q+1Izuop0dSgYaK2LaoSa5u7Rd0458eoMiOQTV1ZpklBKn8NmGNdkOuMqnUv6Szpy73Mshy1P9QQeo+538EWvi+k/QKSLXTnFcj6csEUMOQmVZDq3fSLaJaDl5GAM/53uIjmoTJADFcfJNyDgLaIKFbWoebfe2ZQeS5PtgCI/S+YjR1Z1v5igaB92sKy5kX26OHmmuGiVKD3FHNeguLn/8i7WW5ejKgMxTVNbp1XWZKW1sQu3yAXa4rl5vASDvsiSz2uvvhU/tLF5DUKO1AHS/VBn7DUEkCnR3JkfUTepriRZZk9vo1RV+N7rH8VlON/DVJJ+kSdbCmEWIJp/wYxtIExg9Bb0S0kfXSiLylZCXzetu8Y20WsYWB66fkC9YR7MxcvmYWuSiy26qadJ73khPtYiJSjhcO4vhJSW8SCFvUYEGYQ49A7YnZB0GbiQUi2YNpt9rCeuyzKochApzzlVW7tkTpqZLs7bt15OjXadnuHonY69LCmIz5pnfvp5n59bUeV2ajyzonHuV9bV3rZXl+d/ZUX6zSHgrfXkOWBZj36vUK88Dyxryjc96bOxpPoaR4P2sWADEXMvldET6112e7obO3uxS13Ak9duzXxKRmjd+qjy3HJWYZQUWWSiK42NuwGta8npUhJmw4iLUkhnqb26c1PTFpTBL8ilUFbrTs5ISk37xtWg9sj3rd+yrmnVkleqokIkv0vT1E/BYpSAEH0cA8lKi7nuZrMSQcWDVDllqUyEqik+Mi0yaEMM74qcuACZhi1K7ghZ5PG26NzJdIqHRTXSKms6c3W2nO3YxzrSKhfidaKow3geH0t5hD2eVx5J2XgFfWw8/X/6uNZA"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
coordinator_url = "" #@param {type:"string"}
#@markdown *Optional*: If Google Drive auto-discovery takes time, paste the Cloudflare or Gradio link from the Main coordinator here (e.g. `https://xyz.trycloudflare.com`). Leave blank for auto-discovery.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Coordinator URL: {coordinator_url if coordinator_url else 'Auto-Discover (via Google Drive master.json)'}")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(coordinator_url=coordinator_url, enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
